# History states

Pausing a hierarchical machine and resuming it should usually continue where it left off. A **history pseudo-state** remembers the last active configuration of a compound state:

- **shallow** history restores the last active *direct child* (then enters that child's default sub-state);
- **deep** history restores the exact last active *leaf* at any depth.

The washing machine has `running` (fill, wash with soak/agitate, spin), `paused` and `off`. `resume` targets the history state inside `running`.

```mermaid
stateDiagram-v2
    [*] --> off
    off --> running: start
    running --> paused: pause
    paused --> running: resume (to history)
    running --> off: stop
    paused --> off: stop
    state running {
        [*] --> fill
        fill --> wash: filled
        wash --> spin: washed
        state wash {
            [*] --> soak
            soak --> agitate: soaked
        }
    }
```

In [1]:
from petrilab.fsm.history import WasherDeep, WasherShallow


def ids(sm):
    return sorted(s.id for s in sm.configuration)


def run(sm, *events):
    for e in events:
        sm.send(e)
    return sm


deep = run(WasherDeep(), "start", "filled", "soaked", "pause")
print("paused:", ids(deep))
run(deep, "resume")
ids(deep)

paused: ['paused']


['agitate', 'running', 'wash']

We reached `agitate` inside `wash`, paused, and resumed. Deep history restores exactly `running` / `wash` / `agitate`.

In [2]:
shallow = run(WasherShallow(), "start", "filled", "soaked", "pause", "resume")
ids(shallow)

['running', 'soak', 'wash']

With shallow history, `wash` is remembered as the direct child, but its own sub-state is reset to the default `soak`. That is the only difference between the two kinds.

## A simple (non-compound) child
When the remembered child is a plain state such as `spin`, both history types agree.

In [3]:
for cls in (WasherDeep, WasherShallow):
    sm = run(cls(), "start", "filled", "soaked", "washed", "pause", "resume")
    print(cls.__name__, ids(sm))

WasherDeep ['running', 'spin']
WasherShallow ['running', 'spin']


Both restore `spin`, since there is nothing deeper to restore.

## Stopping and invalid events

In [4]:
from statemachine.exceptions import TransitionNotAllowed

print(ids(run(WasherDeep(), "start", "pause", "stop")))
try:
    WasherDeep().send("resume")
except TransitionNotAllowed:
    print("resume is rejected when not paused")

['off']
resume is rejected when not paused


`stop` works from `paused`, and `resume` is only valid from `paused`.